Traditional RAG → chunk → embed → cosine similarity → retrieve

PageIndex RAG → build tree → LLM reasons over tree → retrieve exact sections

# Section 1: Install & Setup

**What we do here:**

- Install PageIndex SDK + OpenAI
- Load API keys from .env
- Initialize both clients

In [2]:
import os
import json
import time

from dotenv import load_dotenv
load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY    = os.getenv("GROQ_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Groq key loaded:   ", "✅" if GROQ_API_KEY    else "❌ Missing!")

PageIndex key loaded: ✅
Groq key loaded:    ✅


In [3]:
from pageindex import PageIndexClient
from openai import OpenAI

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = OpenAI(
                    api_key=GROQ_API_KEY,
                    base_url="https://api.groq.com/openai/v1",)

print("✅ PageIndex client ready")
print("✅ Groq client ready")

✅ PageIndex client ready
✅ Groq client ready


# Section 2: Upload & Index a PDF

**What happens here:**

- Upload your PDF to the PageIndex cloud
- PageIndex uses an LLM to read the document structure
- Builds a hierarchical tree index (like a smart Table of Contents)
- Returns a doc_id for all future operations


**Why NO chunking?**

Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.

In [4]:
PDF_PATH = "./TCP Sample Demo.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("(Save this ID — you'll use it throughout the notebook)")

📤 Uploading: ./TCP Sample Demo.pdf
✅ Uploaded!
📋 Document ID: pi-cmss09f0d02d301qtgpn4ipkl
(Save this ID — you'll use it throughout the notebook)


In [5]:
print("⏳ Building tree index...")
print("(This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
(This runs once per document — the index is cached for reuse)
   Status: completed

✅ Tree index ready!


# Section 3: Inspect the Tree Structure

### What the tree looks like:

```
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

Each node has:

- node_id — unique ID used during retrieval
- title — section heading
- page_index — page number in original PDF
- text — section summary (when node_summary=True)
- nodes — child sections (nested)

This structure is what the LLM reasons over during retrieval.

In [6]:
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 8

 Raw tree (first node):
{
  "title": "TRANSMISSION CONTROL PROTOCOL",
  "node_id": "0000",
  "page_index": 1,
  "summary": "# TRANSMISSION CONTROL PROTOCOL\n\nDARPA INTERNET PROGRAM\n\nPROTOCOL SPECIFICATION\n\nSeptember 1981\n\nprepared for\n\nDefense Advanced Research Projects Agency\n\nInformation Processing Techniques Office\n\n1400 Wilson Boulevard\n\nArlington, Virginia 22209\n\nby\n\nInformation Sciences Institute\n\nUniversity of Southern California\n\n4676 Admiralty Way\n\nMarina del Rey, California 90291\n\nSeptember 1981\n\nTransmission Control Protocol\n\nhttps://datatracker.ietf.org/doc/html/rfc793\n\n1/175\n\n8/13/26, 4:00 PM\n\nRFC 793 - Transmission Control Protocol\n",
  "text": "# TRANSMISSION CONTROL PROTOCOL\n\nDARPA INTERNET PROGRAM\n\nPROTOCOL SPECIFICATION\n\nSeptember 1981\n\nprepared for\n\nDefense Advanced Research Projects Agency\n\nInformation Processing Techniques Office\n\n1400 Wilson Boulevard\n\nArlington, Virginia 22209\n\nby\n\

In [7]:
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("Full Document Structure:\n")
print_tree(pageindex_tree)

Full Document Structure:

[0000] TRANSMISSION CONTROL PROTOCOL  (p.1)
[0001] TABLE OF CONTENTS  (p.2)
[0002] PREFACE  (p.5)
[0003] 1. INTRODUCTION  (p.7)
  └─ [0004] 1.1. Motivation  (p.7)
  └─ [0005] 1.2. Scope  (p.8)
  └─ [0006] 1.3. About this Document  (p.8)
  └─ [0007] 1.4. Interfaces  (p.10)
  └─ [0008] 1.5. Operation  (p.10)
[0009] 2. PHILOSOPHY  (p.18)
  └─ [0010] 2.1. Elements of the Internetwork System  (p.18)
  └─ [0011] 2.2. Model of Operation  (p.18)
  └─ [0012] 2.3. The Host Environment  (p.20)
  └─ [0013] 2.4. Interfaces  (p.22)
  └─ [0014] 2.5. Relation to Other Protocols  (p.22)
  └─ [0015] 2.6. Reliable Communication  (p.22)
  └─ [0016] 2.8. Data Communication  (p.28)
[0017] 3. FUNCTIONAL SPECIFICATION  (p.34)
  └─ [0018] 3.1. Header Format  (p.34)
  └─ [0019] 3.2. Terminology  (p.42)
    └─ [0020] 3.2. TCP Connection Variables and State Definitions  (p.42)
    └─ [0021] TCP Segment Variables and Connection State Transitions  (p.46)
    └─ [0022] 3.3. Sequence Numbers

In [8]:
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"Total nodes in tree: {total}")
print("Each node = one retrievable section of the document")

Total nodes in tree: 63
Each node = one retrievable section of the document


# Section 4: LLM Tree Search — The Core of PageIndex

This is where PageIndex fundamentally differs from vector RAG.

## Vector RAG retrieval:
`query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks`

Problem: finds what's similar, not what's relevant

## PageIndex retrieval:
`query + tree → LLM reasons → "node 0007 and 0008 contain the answer"`

Advantage: LLM understands document structure, context, and intent

In [15]:
def llm_tree_search(query: str, tree: list, model: str = "llama-3.1-8b-instant") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [17]:
query = "What is the covered in Protocol Making?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the covered in Protocol Making?

LLM Reasoning:
{'step1': "Understanding the query: 'What is the covered in Protocol Making?' means we are looking for a section that talks about the purpose or scope of a protocol.", 'step2': 'Looking at the table of contents, we identify sections 1.1, 1.3, and 1.4. These titles seem relevant to the scope of the protocol.', 'step3': "Considering section 1.4 'Interfaces' seems more related to the technical aspects of the protocol rather than its purpose or scope, so it might not be the most relevant.", 'step4': "Section 1 is about introducing the protocol, which is likely to cover its purpose or scope. Hence, we can narrow down our search to section 1, specifically node ID 0003 '1. INTRODUCTION'."}

Selected Node IDs: ['0003']


# Section 5: Full End-to-End RAG Pipeline

**3 steps:**

1. **Tree Search** → LLM picks relevant node_ids
2. **Retrieve** → Fetch the actual section content from those nodes
3. **Generate** → LLM writes a grounded answer with page citations

**What makes this better than vector RAG:**

- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly which section the answer comes from
- No hallucination from irrelevant chunks

In [18]:
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [19]:
def generate_answer(query: str, nodes: list, model: str = "llama-3.1-8b-instant") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
                Answer the question using ONLY the provided context.
                For every claim you make, cite the section title and page number in parentheses.
                Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [20]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [21]:
answer = vectorless_rag(
    query="What are the things inside a TCP Protocol?",
    tree=pageindex_tree
)

🔍 Query: What are the things inside a TCP Protocol?

🧠 Reasoning: ["Looking at the document tree for the query 'What are the things inside a TCP Protocol?'", "Starting with the root node, which is the 'TRANSMISSION CONTROL PROTOCOL' document (node_id = '0000')", 'Checking its children, and looking for nodes relevant to TCP Protocol components', "Found the '1. INTRODUCTION' node (node_id = '0003'), which is likely to contain information about TCP Protocol components", "Exploring the children of '1. INTRODUCTION' node, and found nodes such as '1.4. Interfaces' (node_id = '0007'), '1.5. Operation' (node_id = '0008'), and '3. FUNCTIONAL SPECIFICATION' (node_id = '0017') that might provide details about TCP protocol operations"]...
🎯 Retrieved node IDs: ['0003', '0007', '0008', '0017']
📄 Sections found: ['1. INTRODUCTION', '1.4. Interfaces', '1.5. Operation', '3. FUNCTIONAL SPECIFICATION']

📝 Answer:
Based on the provided context, the TCP Protocol primarily consists of the following things:

In [22]:
test_queries = [
    "What are the types of protocol exist?",
    "What is TCP the best and usefull?",
    "Summarize the TCP Flow Control?",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: What are the types of protocol exist?
A: Based on the provided context, the types of protocols mentioned are not explicitly listed in the given sections. However, it is stated in the "PREFACE" that the document is a "PROTOCOL SPECIFICATION" and in "Functional Specification" that this "RFC defines a protocol (TCP)".

(PREFACE, Page 1, "PREF...
-------------------------------------------------------

Q: What is TCP the best and usefull?
A: TCP is effective in managing window information to encourage or discourage transmissions, depending on the available buffer space (Managing the Window, Page 88). Indicating a large window encourages transmissions, while a small window may introduce a round trip delay (Managing the Window, Page 88). ...
-------------------------------------------------------

Q: Summarize the TCP Flow Control?
A: TCP Flow Control is a mechanism to prevent one side from overwhelming the other with too many packets. The sender sends data in segments, and the receiver 